In [5]:
!pip3 install playwright
!python3 -m playwright install chromium

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
(node:38149) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
162.3 MiB [                    ] 0% 0.0s162.3 MiB [                    ] 0% 136.9s162.3 MiB [                    ] 0% 119.0s162.3 MiB [                    ] 0% 95.0s162.3 MiB [                    ] 0% 78.3s162.3 MiB [                    ] 0% 62.4s162.3 MiB [                    ] 0% 46.4s162.3 MiB [                    ] 0% 33.5s162.3 MiB [                    ] 0% 26.7s162.3 MiB [                    ] 0% 19.1s162.3 MiB [                    ] 1% 15.4s162.3 MiB [                    ] 1% 11.

In [12]:
import asyncio
from pathlib import Path
from playwright.async_api import async_playwright

async def html_to_single_page_poster_pdf(
    html_path: str,
    output_pdf: str,
    poster_selector: str = ".poster",   # <-- default to your wrapper
    extra_wait_ms: int = 300,
    scale: float = 1.0
):
    html_path = str(Path(html_path).resolve())
    out_path = Path(output_pdf).resolve()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    url = Path(html_path).as_uri()

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()

        await page.goto(url, wait_until="networkidle")

        # Ensure fonts/images are ready
        await page.evaluate("() => document.fonts ? document.fonts.ready : Promise.resolve()")
        await page.wait_for_timeout(extra_wait_ms)

        # Wait for QR code to render (your HTML uses qrcodejs)
        try:
            await page.wait_for_function(
                "document.querySelector('#qrcode canvas, #qrcode img') !== null",
                timeout=5000
            )
        except:
            pass  # if QR isn't present, continue

        poster = page.locator(poster_selector)
        if await poster.count() == 0:
            await browser.close()
            raise ValueError(f"Could not find poster element with selector: {poster_selector}")

        # Measure poster
        box = await poster.bounding_box()
        if not box:
            await browser.close()
            raise RuntimeError("Poster element has no bounding box (maybe display:none?).")

        width_px = int(box["width"] * scale)
        height_px = int(box["height"] * scale)

        # Make viewport big enough to avoid clipping and reflow
        await page.set_viewport_size({"width": width_px, "height": min(height_px, 5000)})
        await page.wait_for_timeout(100)

        # Re-measure after viewport update
        box = await poster.bounding_box()
        width_px = int(box["width"] * scale)
        height_px = int(box["height"] * scale)

        # Print-friendly CSS: no margins, exact colors
        await page.add_style_tag(content="""
            @page { margin: 0 !important; }
            html, body { margin: 0 !important; padding: 0 !important; }
            * { -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }
        """)

        await page.pdf(
            path=str(out_path),
            print_background=True,
            prefer_css_page_size=False,
            width=f"{width_px}px",
            height=f"{height_px}px",
            margin={"top": "0px", "right": "0px", "bottom": "0px", "left": "0px"},
            page_ranges="1",
            scale=1.0
        )

        await browser.close()
        print(f"Saved: {out_path}  (single page crop: {width_px}px x {height_px}px)")

# Jupyter usage:

In [14]:
import os 

In [16]:
posters = os.listdir("org")

In [19]:
from pathlib import Path

# make sure output dir exists
Path("pdf").mkdir(parents=True, exist_ok=True)
for poster in posters:
    await html_to_single_page_poster_pdf(
        html_path=f"org/{poster}",
        output_pdf=f"pdf/{poster.replace('.html', '.pdf')}",
        poster_selector=".poster",   # <-- important
        scale=1.0
    )

Saved: /Users/wajid/Desktop/ACM/newWebsite/csrp-rowan.github.io/posters/pdf/5.pdf  (single page crop: 794px x 1123px)
Saved: /Users/wajid/Desktop/ACM/newWebsite/csrp-rowan.github.io/posters/pdf/4.pdf  (single page crop: 794px x 1123px)
Saved: /Users/wajid/Desktop/ACM/newWebsite/csrp-rowan.github.io/posters/pdf/3.pdf  (single page crop: 794px x 1123px)
Saved: /Users/wajid/Desktop/ACM/newWebsite/csrp-rowan.github.io/posters/pdf/2.pdf  (single page crop: 794px x 1777px)
Saved: /Users/wajid/Desktop/ACM/newWebsite/csrp-rowan.github.io/posters/pdf/1.pdf  (single page crop: 714px x 1270px)


In [13]:
await html_to_single_page_poster_pdf("org/1.html", "pdf/1.pdf", poster_selector=".poster")


Saved: /Users/wajid/Desktop/ACM/newWebsite/csrp-rowan.github.io/posters/pdf/1.pdf  (single page crop: 714px x 1270px)
